# Agents example 1: Our first StateGraph

Run these cells from top to bottom in Google Colab. This first step installs LangGraph and checks its imports. No API key, GPU, or external service connection is required. The installation downloads packages from PyPI.

LangGraph installs its required dependencies automatically. We will add a model provider package later if the example needs one.

In [ ]:
# Install LangGraph in the current Colab runtime.
%pip install -q "langgraph>=1.0,<2.0"

## Check that installation worked

`StateGraph` lets us define a graph. `START` and `END` mark its beginning and end. `TypedDict` describes the state passed between nodes and comes with Python.

This cell only checks imports; we will build the graph next.

In [ ]:
from importlib.metadata import version
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph version:", version("langgraph"))
print("Setup complete! Ready to build our first graph.")

## Colab runtime notes

Run the installation cell again whenever Colab starts a fresh runtime. If Colab asks you to restart the runtime after installation, restart it and rerun the import cell.

[Official installation guide](https://docs.langchain.com/oss/python/langgraph/install)

## 1. What is a StateGraph?

A **StateGraph** is a workflow whose steps share data called **state**.

- **State:** a dictionary containing the current data.
- **Node:** a Python function that reads state and returns updates.
- **Edge:** a connection that determines which node runs next.

We will start with the number 4, add 2, then multiply by 3:

```text
START -> add_two -> multiply_by_three -> END
   4         6               18
```

START and END are special markers, not functions we write. This example uses ordinary Python arithmetic: no model or API calls.


## 2. Describe the state

Our shared dictionary has just one field, `number`. `TypedDict` describes its expected structure for readers and type checkers; it does not validate values at runtime.


In [ ]:
class NumberState(TypedDict):
    number: int


## 3. Define two nodes

Each node receives the current state and returns a dictionary containing its updates. For this simple field, LangGraph replaces the old number with the returned number. We do not change the input dictionary directly.

Defining these functions does not run them yet.


In [ ]:
def add_two(state: NumberState):
    return {"number": state["number"] + 2}


def multiply_by_three(state: NumberState):
    return {"number": state["number"] * 3}


## 4. Build and compile the graph

`add_node` gives each function a name in the graph. `add_edge` connects the steps in order. `compile()` turns our definition into a runnable graph; it does not execute it yet.


In [ ]:
builder = StateGraph(NumberState)

builder.add_node("add_two", add_two)
builder.add_node("multiply_by_three", multiply_by_three)

builder.add_edge(START, "add_two")
builder.add_edge("add_two", "multiply_by_three")
builder.add_edge("multiply_by_three", END)

graph = builder.compile()


## 5. Run the graph

`invoke()` takes the initial state, executes the connected nodes, and returns the final state. The answer is `(4 + 2) * 3 = 18`.


In [ ]:
initial_state = {"number": 4}
final_state = graph.invoke(initial_state)
print("Initial state:", initial_state)
print("Final state:", final_state)

assert final_state == {"number": 18}


Expected output:

```text
Initial state: {'number': 4}
Final state: {'number': 18}
```

## 6. See the updates from each node

This starts a new run from 4. `stream_mode="updates"` shows the changes returned by each node as it finishes.


In [ ]:
for update in graph.stream({"number": 4}, stream_mode="updates"):
    print(update)


Expected output:

```text
{'add_two': {'number': 6}}
{'multiply_by_three': {'number': 18}}
```

## Try it yourself

Change `starting_number` below and predict the result before running. For 10, the result should be 36. Each invocation starts with the state you supply; it does not reuse the previous result.


In [ ]:
starting_number = 10
print(graph.invoke({"number": starting_number}))


The node functions define **what happens**, edges define **the order**, and state carries **the data**. This graph has one fixed path. Later, graphs can also include branches and loops.

[Official StateGraph reference](https://reference.langchain.com/python/langgraph/graph/state/StateGraph)


# Example 2: A dialogue that keeps earlier messages

In the number example, every new `number` replaced the old value. A conversation needs a different rule: keep earlier messages and add the next one.

A **reducer** tells LangGraph how to combine an existing field with a node's update. We will use `add_messages` for the `messages` field.

```text
START -> assistant_greeting -> student_question -> assistant_answer -> END
```

The opening user message enters with the initial state. Three nodes then add three more messages. This is a scripted teaching dialogue: both speakers are simulated by Python functions, with no model or API calls.


## 1. Define a state with a message reducer

`Annotated` attaches the reducer to the field. `BaseMessage`, `HumanMessage`, and `AIMessage` are local message containers from a dependency installed with LangGraph. Creating an `AIMessage` does not call an AI model.

`add_messages` appends messages with new IDs. If an update has the same ID as an existing message, it replaces that message. Here every message is new, so the history grows.


In [ ]:
from typing import Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

class DialogueState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


## 2. Write the dialogue nodes

Each node returns **only the new message**, not a copy of the whole conversation. The reducer combines it with the existing history. The final node reads the latest question from state and quotes it in its response.


In [ ]:
def assistant_greeting(state: DialogueState):
    return {"messages": [AIMessage(content="Hello! Let us learn about LangGraph.")]}


def student_question(state: DialogueState):
    return {"messages": [HumanMessage(content="What is a node?")]}


def assistant_answer(state: DialogueState):
    question = state["messages"][-1].content
    answer = f'You asked: "{question}" A node is a Python function that reads state and returns updates.'
    return {"messages": [AIMessage(content=answer)]}


## 3. Connect and run the dialogue

This helper builds the same sequence for a supplied state schema. We will reuse it below to compare behavior with and without a reducer.


In [ ]:
def build_dialogue(state_schema):
    dialogue_builder = StateGraph(state_schema)
    dialogue_builder.add_node("assistant_greeting", assistant_greeting, input_schema=state_schema)
    dialogue_builder.add_node("student_question", student_question, input_schema=state_schema)
    dialogue_builder.add_node("assistant_answer", assistant_answer, input_schema=state_schema)
    dialogue_builder.add_edge(START, "assistant_greeting")
    dialogue_builder.add_edge("assistant_greeting", "student_question")
    dialogue_builder.add_edge("student_question", "assistant_answer")
    dialogue_builder.add_edge("assistant_answer", END)
    return dialogue_builder.compile()


dialogue_graph = build_dialogue(DialogueState)

def initial_dialogue():
    return {"messages": [HumanMessage(content="Hi! I am a student.")]}


dialogue_result = dialogue_graph.invoke(initial_dialogue())
for message in dialogue_result["messages"]:
    speaker = "Student" if message.type == "human" else "Assistant"
    print(f"{speaker}: {message.content}")

assert len(dialogue_result["messages"]) == 4


Expected dialogue:

```text
Student: Hi! I am a student.
Assistant: Hello! Let us learn about LangGraph.
Student: What is a node?
Assistant: You asked: "What is a node?" A node is a Python function that reads state and returns updates.
```

## 4. Watch the state grow

This starts a fresh run. Unlike the earlier `updates` stream, `stream_mode="values"` shows the full state, including the initial state, so we can count the accumulated messages.


In [ ]:
message_counts = []
for snapshot in dialogue_graph.stream(initial_dialogue(), stream_mode="values"):
    messages = snapshot["messages"]
    message_counts.append(len(messages))
    print(f'{len(messages)} message(s) | latest: {messages[-1].content}')

assert message_counts == [1, 2, 3, 4]


## 5. What happens without the reducer?

Now run the same nodes with a plain list field. Each node still returns one new message, but the default update rule **replaces** the list. Only the final assistant response survives.


In [ ]:
class OverwritingDialogueState(TypedDict):
    messages: list[BaseMessage]


overwriting_graph = build_dialogue(OverwritingDialogueState)
overwriting_result = overwriting_graph.invoke(initial_dialogue())

print("With add_messages:", len(dialogue_result["messages"]), "messages")
print("Without a reducer:", len(overwriting_result["messages"]), "message")
print("Only remaining message:", overwriting_result["messages"][0].content)

assert len(overwriting_result["messages"]) == 1


## What to remember

- Reducers apply to individual state fields. A plain field is replaced; our annotated message field accumulates new messages.
- Each node returns just its update. LangGraph applies the reducer.
- The history is preserved **during this run**. A fresh invocation with a new initial state starts a new dialogue. We have not added a checkpointer or memory between sessions.
- This demonstration uses scripted speakers. To accept a real student's reply, we could add an input step later.

Try changing the student's question in `student_question`, then rerun the cells from that definition onward. The final answer is scripted, so change its explanation too if you ask a different question.

[Official add_messages reference](https://reference.langchain.com/python/langgraph/graph/message/add_messages)
